<a href="https://colab.research.google.com/github/JWang-5779299/AnalyticsEngineering2026/blob/main/4_02_dbt_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 4.02 dbt Workshop

**You do not need to write any code in this notebook.** A dbt project for the bank's card-spend data has been built for you. Your job is to run the cells in order, look at what comes out, and answer the questions in plain English.

- *Core* questions come first in each step. *Stretch* questions are for if you have time.
- Aim for a sentence or two per answer. Use the names you can see on screen (tables, tests, models).
- Keep your answers: they are good practice for the reflection and the test log in the assignment.

The case: our retail bank has three messy source files (card transactions, customer records, exchange rates) and wants trusted monthly spend tables for its dashboards. Now we are doing the work we did previously on paper, but now with `dbt`.

**YOU DO NOT NEED TO USE dbt FOR YOUR INDIVIDUAL ASSIGNMENT AND IT CAN BE TRICKY TO CONFIGURE.**

## Step 0: set up (about 2 minutes)

In [1]:
# Install dbt and DuckDB. Versions are pinned so everyone gets the same behaviour.
!pip install -q dbt-core==1.12.6 dbt-duckdb==1.11.0 duckdb==2.0.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 1.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.2/118.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.9/144.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.3/177.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.7/442.7 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.5/295.5 kB 18.1 MB/s eta

In [2]:
# Get the project (a folder called card_spend_dbt).
import os, zipfile, urllib.request

# Corrected the URL to the direct asset download link instead of the GitHub release page
DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dagsession/card_spend_dbt.zip"

# If a bad/corrupted zip file was already downloaded, remove it to force a clean re-download
if os.path.exists("card_spend_dbt.zip"):
    try:
        with zipfile.ZipFile("card_spend_dbt.zip") as z:
            pass
    except zipfile.BadZipFile:
        print("Removing invalid zip file before re-downloading...")
        os.remove("card_spend_dbt.zip")

if not os.path.exists("card_spend_dbt.zip"):
    if not DATA_URL:
        raise RuntimeError("DATA_URL is empty. Ask Michael for the link, or upload card_spend_dbt.zip to this notebook.")
    urllib.request.urlretrieve(DATA_URL, "card_spend_dbt.zip")

with zipfile.ZipFile("card_spend_dbt.zip") as z:
    z.extractall(".")
print("Project files:", sorted(os.listdir("card_spend_dbt")))

Project files: ['README.md', 'dbt_project.yml', 'models', 'profiles.yml', 'seeds', 'tests']


In [3]:
# A small helper so you can look inside the tables dbt builds. You do not need to read it.
import json, duckdb, pandas as pd
pd.set_option("display.width", 160)

def sql(query):
    con = duckdb.connect("card_spend_dbt/card_spend.duckdb", read_only=True)
    try:
        return con.sql(query).df()
    finally:
        con.close()

def show(path):
    print("=" * 8, path, "=" * 8)
    print(open("card_spend_dbt/" + path).read())

## Step 1: a model is a SELECT

In dbt, each table is defined by a file containing one SELECT statement. Here are two of them: one from the silver layer and one from the gold layer.

In [4]:
show("models/silver/silver_cards.sql")
show("models/gold/gold_spend_by_month_segment.sql")

======== models/silver/silver_cards.sql ========
-- One row per card: the most recent customer record, with a tidy segment name.

with ranked as (

    select
        card_id,
        customer_id,
        upper(substr(trim(segment), 1, 1)) || lower(substr(trim(segment), 2)) as segment,
        account_status,
        row_number() over (partition by card_id order by snapshot_date desc) as rn
    from {{ ref('bronze_customer_master') }}

)

select card_id, customer_id, segment, account_status
from ranked
where rn = 1

======== models/gold/gold_spend_by_month_segment.sql ========
-- Posted card spend in GBP by month and customer segment.
select
    month,
    segment,
    round(sum(amount_gbp), 2) as spend_gbp,
    count(*)                  as transactions
from {{ ref('silver_transactions') }}
where status = 'POSTED'
group by month, segment
order by month, segment



### Q1 (core)

Find `{{ ref('bronze_customer_master') }}` in `silver_cards.sql`. What is it telling dbt, and what does it mean for the order in which things are built?

**Your answer:** *(double-click here and write a sentence or two)*

It tells dbt that silver_cards depends on the bronze_customer_master model. This means bronze_customer_master must be built before silver_cards runs.

### Q2 (core)

Neither file contains `CREATE TABLE`. So where does the table come from?

**Your answer:** *(double-click here and write a sentence or two)*

dbt automatically creates the table from the result of the SELECT statement when we run the model.

## Step 2: build everything

One command builds the whole project: it loads the three raw files (bronze), builds the silver and gold tables, and runs the tests. Read the output as it scrolls past.

In [5]:
!cd card_spend_dbt && dbt build --profiles-dir . --no-use-colors

10:39:49  Running with dbt=1.12.6
10:39:50  Registered adapter: duckdb=1.11.0
10:39:50  Unable to do partial parsing because saved manifest not found. Starting full parse.
10:39:54  Found 7 models, 3 seeds, 13 data tests, 500 macros
10:39:54  
10:39:55  Concurrency: 1 threads (target='dev')
10:39:55  
10:39:55  1 of 23 START seed file main.bronze_card_transactions .......................... [RUN]
10:39:56  1 of 23 OK loaded seed file main.bronze_card_transactions ...................... [INSERT 28829 in 1.47s]
10:39:56  2 of 23 START seed file main.bronze_customer_master ............................ [RUN]
10:39:56  2 of 23 OK loaded seed file main.bronze_customer_master ........................ [INSERT 5600 in 0.20s]
10:39:56  3 of 23 START seed file main.bronze_fx_rates ................................... [RUN]
10:39:56  3 of 23 OK loaded seed file main.bronze_fx_rates ............................... [INSERT 198 in 0.05s]
10:39:56  4 of 23 START test accepted_values_bronze_card_transac

### Q3 (core)

(a) In what order did dbt do things, and did anyone tell it that order? (b) Read the last line: how many tests passed, warned, errored, and how many steps were skipped?

**Your answer:** *(double-click here and write a sentence or two)*

(a) dbt ran the models in dependency order: bronze first, then silver, then gold. It worked out this order automatically from the `ref()` references, so no one manually specified the sequence.

(b) 21 tests passed, 2 tests warned, 0 tests errored, and 0 steps were skipped.

## Step 3: look inside the tables

The tables dbt built are in a small DuckDB database. Run these cells to look at them.

In [6]:
sql("""
select 'bronze_card_transactions: all rows' as what, count(*) as n from bronze_card_transactions
union all select 'bronze_card_transactions: distinct rows', count(*) from (select distinct * from bronze_card_transactions)
union all select 'silver_transactions_latest: one row per transaction', count(*) from silver_transactions_latest
union all select 'silver_cards: one row per card', count(*) from silver_cards
""")

,what,n
0,bronze_card_transactions: all rows,28829
1,bronze_card_transactions: distinct rows,27560
2,silver_transactions_latest: one row per transa...,15126
3,silver_cards: one row per card,400


In [7]:
sql("select * from silver_transactions_latest where status = 'POSTED' limit 5")

,txn_id,ref_txn_id,card_id,merchant_clean,amount,currency,status,ts,file_date
0,T000029,None,C1108,COSTA COFFEE 3593,2.18,GBP,POSTED,2026-01-01 08:01:54,2026-01-04
1,T000037,None,C1010,SHELL 1760,15.95,GBP,POSTED,2026-01-01 08:28:37,2026-01-04
2,T000065,None,C1228,TFL TRAVEL CH,6.16,GBP,POSTED,2026-01-01 12:34:32,2026-01-02
3,T000066,None,C1081,GREGGS 5015,9.66,GBP,POSTED,2026-01-01 12:41:03,2026-01-04
4,T000067,None,C1296,TFL TRAVEL CH,10.90,GBP,POSTED,2026-01-01 12:48:07,2026-01-02


In [8]:
sql("select segment, count(*) as transactions from silver_transactions group by segment order by segment")

,segment,transactions
0,Premium,4712
1,Standard,10264
2,Unknown,150


In [9]:
sql("select * from gold_spend_by_month_segment")

,month,segment,spend_gbp,transactions
0,2026-01,Premium,52759.63,1610
1,2026-01,Standard,108369.12,3445
2,2026-01,Unknown,1784.22,59
3,2026-02,Premium,43682.06,1413
4,2026-02,Standard,97553.90,3088
5,2026-02,Unknown,1344.12,37
6,2026-03,Premium,46412.98,1477
7,2026-03,Standard,97367.59,3212
8,2026-03,Unknown,1441.57,47


### Q4 (core)

The raw transaction file has 28,829 rows but `silver_transactions_latest` has 15,126. Give two reasons where the rows went.

**Your answer:** *(double-click here and write a sentence or two)*

1. Some rows were duplicates in the raw bronze data and were removed.
2. Some transactions were filtered out because they were not POSTED or were invalid records.

### Q5 (core)

150 transactions have the segment `Unknown`. Why? Is this a bug?

**Your answer:** *(double-click here and write a sentence or two)*

These transactions have a `card_id` that cannot match any customer record in `silver_cards`. This is not necessarily a bug; it just means the customer information for those cards is missing.

### S1 (stretch)

A manager looks at the gold table and says: "Unknown is tiny, just delete it." What do you say?

**Your answer:** *(double-click here and write a sentence or two)*

## Step 4: what dbt actually ran

You wrote `{{ ref(...) }}` in the model. dbt turns that into real SQL before it runs. This is the compiled version of `silver_cards`.

In [ ]:
show("target/compiled/card_spend/models/silver/silver_cards.sql")

### Q6 (core)

Compare it with the original `silver_cards.sql`. What did `{{ ref('bronze_customer_master') }}` turn into?

**Your answer:** *(double-click here and write a sentence or two)*

## Step 5: lineage

dbt records every dependency, so it can draw the lineage graph for you. Compare it with the DAG you drew on paper in part 1. Don't worry about this code - visualisation is messy in terms of code (Tableau is much easier).

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def draw_lineage(changed=None):
    m = json.load(open("card_spend_dbt/target/manifest.json"))
    G = nx.DiGraph()
    for node, parents in m["parent_map"].items():
        if node.split(".")[0] in ("model", "seed"):
            G.add_node(node.split(".")[-1])
            for p in parents:
                if p.split(".")[0] in ("model", "seed"):
                    G.add_edge(p.split(".")[-1], node.split(".")[-1])
    for i, gen in enumerate(nx.topological_generations(G)):
        for n in gen:
            G.nodes[n]["layer"] = i
    # Fixed top-to-bottom order inside each layer, so arrows do not cross boxes.
    order = ["bronze_card_transactions", "bronze_customer_master", "bronze_fx_rates",
             "silver_transactions_latest", "silver_cards", "silver_fx_daily", "silver_transactions",
             "gold_spend_by_month_segment", "gold_reversals_by_month", "gold_foreign_currency_by_month"]
    rank = {n: order.index(n) if n in order else 99 for n in G}
    pos = {}
    for layer in sorted({G.nodes[n]["layer"] for n in G}):
        members = sorted([n for n in G if G.nodes[n]["layer"] == layer], key=lambda n: rank[n])
        for k, n in enumerate(members):
            pos[n] = (layer, (len(members) - 1) / 2 - k)
    hit = nx.descendants(G, changed) | {changed} if changed else set()
    def colour(n):
        if n in hit: return "#F4A261"
        return "#F6E7D6" if n.startswith("bronze") else "#D6E6F5" if n.startswith("silver") else "#FBF4DA"
    fig, ax = plt.subplots(figsize=(14, 5.5))
    nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowsize=18, node_size=5200, width=1.6, edge_color="#7A8CA0")
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=[colour(n) for n in G], node_shape="s", node_size=5200, edgecolors="#1F3A5F")
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=7.5, labels={n: n.replace("_", "\n", 1) if len(n) > 18 else n for n in G})
    ax.set_title("Lineage" + (f": everything downstream of {changed} is highlighted" if changed else ""))
    ax.margins(0.18); ax.axis("off"); plt.show()

draw_lineage()

### Q7 (core)

Compare this graph with the DAG you drew on paper. What matched, and what did you miss or add?

**Your answer:** *(double-click here and write a sentence or two)*

In [ ]:
# Which models would be affected if the exchange-rate feed changed?
!cd card_spend_dbt && dbt ls --profiles-dir . --no-use-colors --resource-type model --select bronze_fx_rates+
draw_lineage(changed="bronze_fx_rates")

### Q8 (core)

The exchange-rate provider changes its file format and breaks `bronze_fx_rates`. Which models are affected, and which are safe?

**Your answer:** *(double-click here and write a sentence or two)*

## Step 6: tests

dbt ran 13 tests. They are written in a YAML file next to the models. First, the results.

In [ ]:
rr = json.load(open("card_spend_dbt/target/run_results.json"))
tests = [(r["unique_id"].split(".")[2], r["status"], r.get("failures")) for r in rr["results"] if r["unique_id"].startswith("test.")]
pd.DataFrame(tests, columns=["test", "status", "rows_or_values_failing"]).sort_values("status", ascending=False).reset_index(drop=True)

In [ ]:
show("models/schema.yml")

### Q9 (core)

Two tests give a WARN. Which two, and what is each one telling us?

**Your answer:** *(double-click here and write a sentence or two)*

In [ ]:
# What does "Got 6 results" on the bronze currency test actually count?
sql("select currency, count(*) as rows from bronze_card_transactions where currency not in ('GBP','EUR','USD','CHF') group by currency order by rows desc")

### Q10 (core)

The bronze currency test reports 6 results. Does that mean 6 bad rows?

**Your answer:** *(double-click here and write a sentence or two)*

### S2 (stretch)

The currency test is set to `warn` on bronze, but the same test on silver has no severity set, so it stops the build (error). Why the difference?

**Your answer:** *(double-click here and write a sentence or two)*

### S3 (stretch)

The `relationships` warning covers 150 transactions. From week 3: should this fail the build, quarantine those rows, or just flag them? Justify your choice.

**Your answer:** *(double-click here and write a sentence or two)*

## Step 7: break it

Real data changes. Here a new transaction arrives in a currency the bank has never seen (JPY). Run the next cell, which adds that one row to the raw file and rebuilds. Watch what happens to each step.

In [ ]:
# Add one new JPY transaction to the raw file, then rebuild.
with open("card_spend_dbt/seeds/bronze_card_transactions.csv", "a") as f:
    f.write("T999999,,C1001,TEST MERCHANT,10.00,JPY,POSTED,2026-03-30 10:00:00,2026-03-30\n")
!cd card_spend_dbt && dbt build --profiles-dir . --no-use-colors 2>&1 | grep -E "FAIL|SKIP|ERROR|WARN|Done"

In [ ]:
# Are the gold tables still there?
sql("select count(*) as rows_in_gold_spend_table from gold_spend_by_month_segment")

### Q11 (core)

Which test failed? Which steps were skipped, and which carried on as normal? Why was the gold layer skipped when its own code is fine?

**Your answer:** *(double-click here and write a sentence or two)*

### Q12 (core)

The gold table still has rows. So is the dashboard now broken, or something else? What would you need in place to notice?

**Your answer:** *(double-click here and write a sentence or two)*

### S4 (stretch)

Is `error` the right setting for this test? What would you change in the pipeline so a new currency does not stop everything?

**Your answer:** *(double-click here and write a sentence or two)*

In [ ]:
# Put things back: restore the original raw file and rebuild.
with zipfile.ZipFile("card_spend_dbt.zip") as z:
    z.extract("card_spend_dbt/seeds/bronze_card_transactions.csv", ".")
!cd card_spend_dbt && dbt build --profiles-dir . --no-use-colors 2>&1 | grep -E "Done"

## Step 8: back to the platform and your assignment

In the lecture, a platform needs to do seven things: ingest and store data; transform it in layers with code under version control; work out the order from dependencies; test data and code; document and show lineage; re-run safely in separate environments; and serve trusted tables.

### Q13 (core)

Which of the seven did dbt cover in this notebook, and which did something else do (or nothing)?

**Your answer:** *(double-click here and write a sentence or two)*

### S5 (stretch)

For your own assignment: name one check you would put on each layer (bronze, silver, gold), and say whether it should warn or stop the build.

**Your answer:** *(double-click here and write a sentence or two)*

**Done.** In the assignment you can use dbt, but you do not have to. Whatever you build, the same ideas apply: layers, a known order, checks that could catch a problem, and a record of what you did when they did.